# A Loss With a Corner In It

**Topic:** Hinge loss and subgradients &nbsp;·&nbsp; **Difficulty:** Medium

Hinge loss has a kink exactly at the margin, so it has no derivative there. That
does not stop gradient descent. It stops *naive* gradient descent. The fix is a
subgradient: any element of the set of valid slopes at the corner will do, and
choosing zero is the standard pick.

Train a linear SVM with subgradient descent, then check the answer against
sklearn's solver.

## The data (created in the next cell)

- `X` `(700, 5)`, `y` `(700,)` with labels in `{-1, +1}` (note: not 0/1).
- `X_test` `(300, 5)`, `y_test` for scoring.
- Objective: `0.5 * lam * ||w||^2 + mean(max(0, 1 - y (Xw + b)))`, `lam = 0.01`.

## Your task

With **NumPy only**:

1. Initialize `w = 0`, `b = 0`. Run 3000 subgradient steps with a decaying step
   size `eta_t = 0.5 / (1 + 0.01 * t)` for `t = 0, 1, ...`:
   - margins `m = y * (X @ w + b)`,
   - active set `A = (m < 1)`,
   - `grad_w = lam * w - mean over all rows of (A * y)[:, None] * X`,
   - `grad_b = -mean over all rows of (A * y)`.
2. Set `answers` to a dict with `"w"` `(5,)`, `"b"` (float),
   `"train_obj"` (the objective at the final `w, b`), `"test_acc"`, and
   `"n_margin_violations"` (int, test points with `m < 1`).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
also checks your test accuracy is within 3 points of sklearn's `LinearSVC`.

In [ ]:
import numpy as np

rng = np.random.default_rng(1010)
mu = np.array([0.9, -0.7, 0.5, 0.0, 0.0])

def make(n, seed):
    r = np.random.default_rng(seed)
    y = r.choice([-1, 1], n)
    X = y[:, None] * mu + r.normal(0, 1.0, (n, 5))
    return X, y.astype(float)

X, y = make(700, 41)
X_test, y_test = make(300, 42)
lam = 0.01
print(f"X {X.shape}, labels in {set(np.unique(y).astype(int).tolist())}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("w", [5], 0.7881879344883604, -1.5272031883680637, 1e-06),
    ("b", [], 0.18509777306982705, -0.030992403669396505, 1e-06),
    ("train_obj", [], 0.24469393821452992, -0.0409710672517826, 1e-06),
    ("test_acc", [], 0.88, -0.14734545303675925, 1e-06),
    ("n_margin_violations", [], 88.0, -14.734545303675926, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "test_acc" in answers:
    from sklearn.svm import LinearSVC
    _ref = LinearSVC(C=1.0 / (lam * len(y)), max_iter=20000).fit(X, y)
    _ra = float((_ref.predict(X_test) == y_test).mean())
    _close = abs(answers["test_acc"] - _ra) < 0.03
    print(f"  {'within 3pts of LinearSVC':<24} {'ok' if _close else 'wrong'}  "
          f"({answers['test_acc']:.3f} vs {_ra:.3f})")
    _ok = _ok and _close

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

At the kink the loss is not differentiable, but `max(0, u)` has subgradient `0`
for `u < 0`, `1` for `u > 0`, and anything in `[0, 1]` at `u = 0`. Taking the
active set as strictly `m < 1` picks the zero subgradient at the corner, which is
both valid and the conventional choice. The step size must decay. With a fixed
step, subgradient descent oscillates around the optimum rather than settling.

```python
import numpy as np

n, d = X.shape
w = np.zeros(d)
b = 0.0

for t in range(3000):
    eta = 0.5 / (1 + 0.01 * t)
    m = y * (X @ w + b)
    A = (m < 1).astype(float)
    gw = lam * w - ((A * y)[:, None] * X).mean(axis=0)
    gb = -float((A * y).mean())
    w = w - eta * gw
    b = b - eta * gb

m = y * (X @ w + b)
train_obj = float(0.5 * lam * (w @ w) + np.maximum(0.0, 1 - m).mean())
mt = y_test * (X_test @ w + b)

answers = {
    "w": w,
    "b": float(b),
    "train_obj": train_obj,
    "test_acc": float((np.sign(X_test @ w + b) == y_test).mean()),
    "n_margin_violations": int((mt < 1).sum()),
}
```